# Revision Robustness Checks for Reviewer Response

The following two cells are **self-contained** and can be run independently in Colab without executing the earlier notebook cells.

- **Appendix Table A3**: OLS robustness check excluding the March–May 2020 COVID-19 oil-price collapse period.
- **Appendix Table A4**: GARCH(1,1) sensitivity check using Student-t errors.

Both cells download the required data directly from Yahoo Finance through `yfinance`, apply the same weekly-frequency and transformation logic, and display the resulting appendix tables as **HTML tables with explicit column headers**. This prevents the column names from being concatenated when the output is copied into Word or Markdown.

No CSV output is required. If a file export is later needed, the optional `to_csv()` lines at the end of each cell can be uncommented.


In [ ]:
# ============================================================
# Appendix Table A3
# Robustness Check Excluding the COVID-19 Oil-Price Collapse Period
# ============================================================
# This cell is self-contained. It can be run independently in Colab.

!pip -q install yfinance pandas numpy statsmodels scipy tabulate

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from IPython.display import display, Markdown

# -----------------------------
# 1. Configuration
# -----------------------------
START = "2019-01-01"
END = "2026-05-01"          # raw-data download endpoint; covers data through April 2026
WAR_DATE = "2022-02-24"
COVID_START = "2020-03-01"
COVID_END = "2020-05-31"

TICKERS = {
    "Oil": "BZ=F",          # Brent crude oil futures
    "Gas": "TTF=F",        # Dutch TTF natural gas futures
    "VIX": "^VIX",         # CBOE Volatility Index
    "DXY": "DX-Y.NYB",     # U.S. Dollar Index
    "RUB": "RUB=X",        # RUB per USD
    "UAH": "UAH=X",        # UAH per USD
    "KRW": "KRW=X",        # KRW per USD
    "JPY": "JPY=X",        # JPY per USD
    "EURUSD": "EURUSD=X",  # Yahoo quote; converted to EUR per USD direction
    "CAD": "CAD=X",        # CAD per USD
    "NOK": "NOK=X",        # NOK per USD
    "KZT": "KZT=X",        # KZT per USD
}

MAIN_CURRENCIES = ["RUB", "UAH", "KRW"]
SUPPLEMENTARY_CURRENCIES = ["JPY", "EUR", "CAD", "NOK", "KZT"]
REGRESSORS = ["Oil", "Gas", "DXY", "VIX"]

# -----------------------------
# 2. Download and transformation
# -----------------------------
raw = yf.download(
    tickers=list(TICKERS.values()),
    start=START,
    end=END,
    interval="1wk",
    auto_adjust=False,
    progress=False,
    group_by="column",
    threads=True,
)

if raw.empty:
    raise RuntimeError("No data were downloaded. Check internet access and ticker availability.")

if isinstance(raw.columns, pd.MultiIndex):
    field = "Adj Close" if "Adj Close" in raw.columns.get_level_values(0) else "Close"
    prices = raw[field].copy()
else:
    field = "Adj Close" if "Adj Close" in raw.columns else "Close"
    prices = raw[[field]].copy()
    prices.columns = list(TICKERS.values())[:1]

reverse = {v: k for k, v in TICKERS.items()}
prices = prices.rename(columns=reverse)
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices = prices.sort_index().ffill().bfill()

# Convert EUR/USD into USD/EUR direction before log differencing,
# consistent with the "local-currency units per U.S. dollar" convention.
if "EURUSD" in prices.columns:
    prices["EUR"] = 1.0 / prices["EURUSD"]
    prices = prices.drop(columns=["EURUSD"])

prices = prices.where(prices > 0)
data = np.log(prices).diff().dropna(how="any")
ordered_cols = [c for c in ["Oil", "Gas", "VIX", "DXY", "RUB", "UAH", "KRW", "JPY", "EUR", "CAD", "NOK", "KZT"] if c in data.columns]
data = data[ordered_cols]

# -----------------------------
# 3. OLS function
# -----------------------------
def fit_ols(df, currency):
    d = df[[currency] + REGRESSORS].dropna()
    if len(d) < len(REGRESSORS) + 10:
        return {
            "Currency": currency,
            "Oil coefficient": np.nan,
            "Oil p-value": np.nan,
            "Adj. R²": np.nan,
            "N": len(d),
            "Status": "Too few observations",
        }

    y = d[currency]
    X = sm.add_constant(d[REGRESSORS], has_constant="add")
    res = sm.OLS(y, X).fit()

    return {
        "Currency": currency,
        "Oil coefficient": res.params.get("Oil", np.nan),
        "Oil p-value": res.pvalues.get("Oil", np.nan),
        "Adj. R²": res.rsquared_adj,
        "N": int(res.nobs),
        "Status": "OK",
    }

def split_periods(df, exclude_covid=False):
    war = pd.Timestamp(WAR_DATE)
    pre = df.loc[df.index < war].copy()
    post = df.loc[df.index >= war].copy()

    if exclude_covid:
        c0 = pd.Timestamp(COVID_START)
        c1 = pd.Timestamp(COVID_END)
        pre = pre.loc[~((pre.index >= c0) & (pre.index <= c1))].copy()

    return [("Prewar", pre), ("Postwar", post)]

# -----------------------------
# 4. Run baseline and COVID-excluded OLS
# -----------------------------
currencies = [c for c in MAIN_CURRENCIES + SUPPLEMENTARY_CURRENCIES if c in data.columns]

rows = []
for spec_name, exclude_covid in [("Baseline", False), ("COVID-excluded", True)]:
    for period, subset in split_periods(data, exclude_covid=exclude_covid):
        for cur in currencies:
            row = fit_ols(subset, cur)
            row["Specification"] = spec_name
            row["Period"] = period
            row["Excluded window"] = f"{COVID_START} to {COVID_END}" if exclude_covid else ""
            rows.append(row)

full_results = pd.DataFrame(rows)
full_results = full_results[
    ["Specification", "Currency", "Period", "Oil coefficient", "Oil p-value", "Adj. R²", "N", "Excluded window", "Status"]
]

appendix_a3 = full_results[full_results["Specification"] == "COVID-excluded"].copy()
appendix_a3 = appendix_a3.drop(columns=["Specification", "Excluded window", "Status"])

# -----------------------------
# 5. Display table output directly in notebook
# -----------------------------
from IPython.display import display, Markdown, HTML

a3_table = appendix_a3.copy().reset_index(drop=True)

# Manuscript-ready numeric formatting
a3_display = a3_table.copy()
a3_display["Oil coefficient"] = a3_display["Oil coefficient"].map(lambda x: f"{x:.3f}")
a3_display["Oil p-value"] = a3_display["Oil p-value"].map(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")
a3_display["Adj. R²"] = a3_display["Adj. R²"].map(lambda x: f"{x:.3f}")
a3_display["N"] = a3_display["N"].astype(int).astype(str)

display(Markdown("### Appendix Table A3. Robustness Check Excluding the COVID-19 Oil-Price Collapse Period"))

html_a3 = a3_display.to_html(index=False, escape=False, border=0)
html_a3 = html_a3.replace(
    '<table border="0" class="dataframe">',
    '<table style="border-collapse:collapse; font-size:13px; width:auto;" border="1">'
)
html_a3 = html_a3.replace(
    '<th>',
    '<th style="border:1px solid #999; padding:4px 8px; text-align:center; background-color:#f2f2f2;">'
)
html_a3 = html_a3.replace(
    '<td>',
    '<td style="border:1px solid #999; padding:4px 8px; text-align:center;">'
)
display(HTML(html_a3))

display(Markdown(
    "**Note.** The March–May 2020 COVID-19 oil-price collapse period is excluded from the prewar sample. "
    "The same OLS specification as the main analysis is used. All variables are weekly log returns or rates of change."
))

# Plain-text Markdown version for copying into a manuscript or response letter
print("\nMarkdown table for copy/paste:\n")
print(a3_display.to_markdown(index=False))

# Optional export, if needed later:
# full_results.to_csv("ols_baseline_vs_covid_excluded.csv", index=False, encoding="utf-8-sig")
# appendix_a3.to_csv("appendix_table_a3_covid_excluded_ols.csv", index=False, encoding="utf-8-sig")


In [ ]:
# ============================================================
# Appendix Table A4
# GARCH(1,1) Robustness Check with Student-t Errors
# ============================================================
# This cell is self-contained. It can be run independently in Colab.

!pip -q install yfinance pandas numpy arch tabulate

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from IPython.display import display, Markdown

# -----------------------------
# 1. Configuration
# -----------------------------
START = "2019-01-01"
END = "2026-05-01"
WAR_DATE = "2022-02-24"

TICKERS = {
    "RUB": "RUB=X",    # RUB per USD
    "UAH": "UAH=X",    # UAH per USD
    "KRW": "KRW=X",    # KRW per USD
}

CURRENCIES = ["RUB", "UAH", "KRW"]
RETURN_SCALE = 100.0    # Scaling improves numerical stability in GARCH estimation.

# -----------------------------
# 2. Download and transformation
# -----------------------------
raw = yf.download(
    tickers=list(TICKERS.values()),
    start=START,
    end=END,
    interval="1wk",
    auto_adjust=False,
    progress=False,
    group_by="column",
    threads=True,
)

if raw.empty:
    raise RuntimeError("No data were downloaded. Check internet access and ticker availability.")

if isinstance(raw.columns, pd.MultiIndex):
    field = "Adj Close" if "Adj Close" in raw.columns.get_level_values(0) else "Close"
    prices = raw[field].copy()
else:
    field = "Adj Close" if "Adj Close" in raw.columns else "Close"
    prices = raw[[field]].copy()
    prices.columns = list(TICKERS.values())[:1]

reverse = {v: k for k, v in TICKERS.items()}
prices = prices.rename(columns=reverse)
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices = prices.sort_index().ffill().bfill()

prices = prices.where(prices > 0)
data = np.log(prices).diff().dropna(how="any")
data = data[[c for c in CURRENCIES if c in data.columns]]

# -----------------------------
# 3. GARCH function
# -----------------------------
def fit_garch_student_t(series, scale=RETURN_SCALE):
    y = series.dropna() * scale

    if len(y) < 50:
        return {
            "omega": np.nan,
            "alpha": np.nan,
            "beta": np.nan,
            "nu": np.nan,
            "LogLik": np.nan,
            "N": len(y),
            "Converged": False,
            "Boundary flag": "Too few observations",
            "Status": "Too few observations",
        }

    try:
        model = arch_model(
            y,
            mean="Constant",
            vol="GARCH",
            p=1,
            o=0,
            q=1,
            dist="t",
            rescale=False,
        )
        res = model.fit(disp="off", show_warning=False, options={"maxiter": 2000})
        params = res.params

        omega = float(params.get("omega", np.nan))
        alpha = float(params.get("alpha[1]", np.nan))
        beta = float(params.get("beta[1]", np.nan))
        nu = float(params.get("nu", np.nan)) if "nu" in params.index else np.nan

        convergence_flag = getattr(res, "convergence_flag", None)
        converged = True if convergence_flag is None else (convergence_flag == 0)

        flags = []
        if np.isfinite(alpha) and alpha <= 1e-4:
            flags.append("alpha near 0")
        if np.isfinite(beta) and beta >= 0.98:
            flags.append("beta near 1")
        if np.isfinite(alpha) and np.isfinite(beta) and alpha + beta >= 0.999:
            flags.append("alpha+beta near/nonstationary")
        if not converged:
            flags.append(f"convergence_flag={convergence_flag}")

        return {
            "omega": omega,
            "alpha": alpha,
            "beta": beta,
            "nu": nu,
            "LogLik": float(res.loglikelihood),
            "N": int(res.nobs),
            "Converged": bool(converged),
            "Boundary flag": "; ".join(flags),
            "Status": "OK",
        }

    except Exception as exc:
        return {
            "omega": np.nan,
            "alpha": np.nan,
            "beta": np.nan,
            "nu": np.nan,
            "LogLik": np.nan,
            "N": len(y),
            "Converged": False,
            "Boundary flag": "Estimation failed",
            "Status": f"Failed: {type(exc).__name__}: {exc}",
        }

def split_periods(df):
    war = pd.Timestamp(WAR_DATE)
    return [
        ("Prewar", df.loc[df.index < war].copy()),
        ("Postwar", df.loc[df.index >= war].copy()),
    ]

# -----------------------------
# 4. Run Student-t GARCH sensitivity check
# -----------------------------
rows = []
for period, subset in split_periods(data):
    for cur in CURRENCIES:
        if cur not in subset.columns:
            continue
        row = fit_garch_student_t(subset[cur])
        row["Currency"] = cur
        row["Period"] = period
        row["Distribution"] = "Student-t"
        row["Scale"] = RETURN_SCALE
        rows.append(row)

appendix_a4 = pd.DataFrame(rows)
appendix_a4 = appendix_a4[
    ["Currency", "Period", "Distribution", "omega", "alpha", "beta", "nu", "LogLik", "N", "Converged", "Boundary flag", "Status"]
]

# -----------------------------
# 5. Display table output directly in notebook
# -----------------------------
from IPython.display import display, Markdown, HTML

a4_table = appendix_a4.copy().reset_index(drop=True)

# Manuscript-ready numeric formatting
a4_display = a4_table.copy()
for col in ["omega", "alpha", "beta", "nu", "LogLik"]:
    a4_display[col] = a4_display[col].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
a4_display["N"] = a4_display["N"].astype(int).astype(str)
a4_display["Converged"] = a4_display["Converged"].astype(str)

display(Markdown("### Appendix Table A4. GARCH Robustness Check with Student-t Errors"))

html_a4 = a4_display.to_html(index=False, escape=False, border=0)
html_a4 = html_a4.replace(
    '<table border="0" class="dataframe">',
    '<table style="border-collapse:collapse; font-size:13px; width:auto;" border="1">'
)
html_a4 = html_a4.replace(
    '<th>',
    '<th style="border:1px solid #999; padding:4px 8px; text-align:center; background-color:#f2f2f2;">'
)
html_a4 = html_a4.replace(
    '<td>',
    '<td style="border:1px solid #999; padding:4px 8px; text-align:center;">'
)
display(HTML(html_a4))

display(Markdown(
    "**Note.** GARCH(1,1) models are estimated using Student-t errors. "
    "Returns are multiplied by 100 before estimation for numerical stability. "
    "Boundary flags indicate estimates that require cautious interpretation."
))

# Plain-text Markdown version for copying into a manuscript or response letter
print("\nMarkdown table for copy/paste:\n")
print(a4_display.to_markdown(index=False))

# Optional export, if needed later:
# appendix_a4.to_csv("appendix_table_a4_student_t_garch.csv", index=False, encoding="utf-8-sig")
